# Setup: bronze, silver and gold

Creates the `bronze` schema and the `bronze.raw_mandi_transactions` table and the `bronze.raw_dim_market`,
`bronze.raw_dim_commodity` and `bronze.raw_dim_date` tables (loaded by `src/load_bronze_dims`), and the `bronze.landing_zone` volume for the source files and the `bronze.archive` volume for the files already loaded, then the `silver` schema and the
`silver.dim_market`, `silver.dim_commodity` and `silver.dim_date` tables (loaded by `src/load_silver_dims` from the bronze seed tables) and the
`silver.mandi_prices_clean` and `silver.mandi_prices_rejected` tables (loaded by `src/load_silver_transactions`) and the
`silver.fact_mandi_prices` table (loaded by `src/load_silver_facts`), then the `gold` schema and the
`gold.mart_price_weekly` table (loaded by `src/load_gold`),
in the workspace's default catalog (`SELECT current_catalog()`), which must already exist.

Then grants read and write access on every bronze volume (`landing_zone`, `archive`) to each principal in
the `volume_users` widget (comma-separated users, groups or service principal IDs; empty grants nothing), so
they can upload source files. USE CATALOG is not granted: it is granted once by hand.

Widgets:
- `volume_users`: comma-separated principals to grant volume access to; set per target in `databricks.yml`.

Safe to re-run: every statement uses `IF NOT EXISTS`, and granting a privilege twice changes nothing.
Note: `IF NOT EXISTS` skips an existing table even if its columns differ; it never alters it.

Raises an error if any step fails, so the run shows as failed.

In [ ]:
from lib_mandi.ddl import (
    create__schema,
    create__table__dim_commodity,
    create__table__dim_date,
    create__table__dim_market,
    create__table__fact_mandi_prices,
    create__table__mandi_prices_clean,
    create__table__mandi_prices_rejected,
    create__table__mart_price_weekly,
    create__table__raw_dim_commodity,
    create__table__raw_dim_date,
    create__table__raw_dim_market,
    create__table__raw_mandi_transactions,
    create__volume,
    grant__volume_access,
)
from lib_mandi.utility import get__default_catalog

dbutils.widgets.text("volume_users", "", "Volume users (comma-separated)")
volume_users = [p.strip() for p in dbutils.widgets.get("volume_users").split(",") if p.strip()]

catalog = get__default_catalog(spark)
print(f"Setting up bronze, silver and gold in catalog: {catalog}")

if not catalog:
    raise RuntimeError("Could not read the default catalog (see the error printed above)")

# (schema, comment, table creators)
layers = [
    (
        "bronze",
        "Raw data stored exactly as received",
        [
            create__table__raw_mandi_transactions,
            create__table__raw_dim_market,
            create__table__raw_dim_commodity,
            create__table__raw_dim_date,
        ],
    ),
    (
        "silver",
        "Cleaned, typed and standardised data, dimensions and facts",
        [
            create__table__dim_market,
            create__table__dim_commodity,
            create__table__dim_date,
            create__table__mandi_prices_clean,
            create__table__mandi_prices_rejected,
            create__table__fact_mandi_prices,
        ],
    ),
    ("gold", "Data marts for the dashboard", [create__table__mart_price_weekly]),
]

for schema, comment, creators in layers:
    if not create__schema(spark, catalog, schema, comment):
        raise RuntimeError(f"Could not create schema {catalog}.{schema} (see the error printed above)")
    print(f"Schema ready: {catalog}.{schema}")

    for create__table in creators:
        table = create__table.__name__.replace("create__table__", "")
        if not create__table(spark, catalog, schema):
            raise RuntimeError(f"Could not create table {catalog}.{schema}.{table} (see the error printed above)")
        print(f"Table ready: {catalog}.{schema}.{table}")

# Landing zone for the source files read by src/load_bronze.
if not create__volume(spark, catalog, "bronze", "landing_zone", "Source files waiting to be loaded into bronze"):
    raise RuntimeError(f"Could not create volume {catalog}.bronze.landing_zone (see the error printed above)")
print(f"Volume ready: {catalog}.bronze.landing_zone")

# Files moved here by src/archive_files once they are loaded.
if not create__volume(spark, catalog, "bronze", "archive", "Source files already loaded into bronze"):
    raise RuntimeError(f"Could not create volume {catalog}.bronze.archive (see the error printed above)")
print(f"Volume ready: {catalog}.bronze.archive")

# Read/write access on every bronze volume for the principals in the volume_users widget.
for volume in ["landing_zone", "archive"]:
    for principal in volume_users:
        if not grant__volume_access(spark, catalog, "bronze", volume, principal):
            raise RuntimeError(
                f"Could not grant access on volume {catalog}.bronze.{volume} to {principal} "
                "(see the error printed above)"
            )
        print(f"Access granted: {principal} on {catalog}.bronze.{volume}")
if not volume_users:
    print("No volume_users set: no volume access granted")

print("Setup complete")